In [1]:
import sys
import os

# Move one directory up (to project root)
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("Added to PYTHONPATH:", project_root)

Added to PYTHONPATH: /vols/cms/mm1221/geant4sim/scripts/validation_new


In [2]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph, radius_neighbors_graph

import pandas as pd
from tqdm import tqdm

def reco_to_sim(hit_energies, rmask, cmask, purities):
    num = 0
    denom = 0
    for i in range(len(hit_energies)):
        if rmask[i]:
            if cmask[i]:
                num += min((1-purities[i])**2, 1) * (hit_energies[i])**2
                denom += hit_energies[i]**2
            else:
                num +=(hit_energies[i])**2
                denom += hit_energies[i]**2
    RtS = num/denom
    return RtS


def build_dataframe(reconstructed_label, loader):
    num_reco = len(reconstructed_label)
    rows = []
    for i, data in enumerate(loader):
        if i >= num_reco:
            break
        CP_ids = data.assoc
        PrimaryEnergies = data.PrimaryEnergies
        reco_ids = reconstructed_label[i]

        hit_energy = np.array(data.x[:,3])
        hit_purity = np.ones_like(hit_energy)
        unique_cp_ids = np.unique(CP_ids)
        unique_reco_ids = np.unique(reco_ids)

        for rid in unique_reco_ids:
            if rid == -1:
                continue
            rmask = np.array((reco_ids == rid))
            for cid in unique_cp_ids:
                cmask = np.array((CP_ids == cid))
                PE = PrimaryEnergies[cid]
                
                cp_energy = hit_energy[cmask].sum()
                reco_energy = hit_energy[rmask].sum()
                shared_energy = hit_energy[rmask & cmask].sum()
                RtS = reco_to_sim(hit_energy, rmask, cmask, hit_purity)
                rows.append({
                    'event_id': i,
                    'cp_id': cid,
                    'reco_id': rid,
                    'cp_energy': cp_energy,
                    'reco_energy': reco_energy,
                    'shared_energy': shared_energy,
                    'RtS': RtS,
                    'PrimaryEnergy' : PE.item()
                })
    df = pd.DataFrame(rows).sort_values(['event_id', 'cp_id', 'reco_id']).reset_index(drop=True)
    return df


    return None

def calc_purity(df, threshold=0.2):
    total_reco = (
        df[['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )
    associated_reco = (
        df.loc[df['RtS'] < threshold, ['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )
    purity = associated_reco / total_reco if total_reco > 0 else 0.0
    return purity

def calc_efficiency(df, threshold=0.5):
    total_cp = (
        df[['event_id', 'cp_id']]
        .drop_duplicates()
        .shape[0]
    )
    associated_cp = (
        df.loc[(df['shared_energy']/df['cp_energy']) > threshold, ['event_id', 'cp_id']]
        .drop_duplicates()
        .shape[0]
    )
    efficiency = associated_cp / total_cp if total_cp > 0 else 0.0
    return efficiency

def calc_merge_rate(df, threshold=0.2):
    # Total number of reconstructed objects
    total_reco = (
        df[['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )

    if total_reco == 0:
        return 0.0

    # Count how many CPs each reco object is associated with
    reco_cp_counts = (
        df.loc[df['RtS'] < threshold, ['event_id', 'reco_id', 'cp_id']]
        .drop_duplicates()
        .groupby(['event_id', 'reco_id'])['cp_id']
        .nunique()
    )

    # Reco objects associated with more than one CP
    merged_reco = (reco_cp_counts > 1).sum()

    merge_rate = merged_reco / total_reco
    return merge_rate

def calc_response(df):
    idx_best = df.groupby(['event_id', 'cp_id'])['shared_energy'].idxmax()
    best_matches = df.loc[idx_best].copy()

    best_matches = best_matches[best_matches['cp_energy'] > 0]
    best_matches['response'] = (
        best_matches['reco_energy'] / best_matches['cp_energy']
    )

    mean_resp = best_matches['response'].mean()
    std_resp  = best_matches['response'].std(ddof=1) 

    return mean_resp, std_resp

def calc_ratio(df):
    total_reco = (
        df[['event_id', 'reco_id']]
        .drop_duplicates()
        .shape[0]
    )
    
    total_cp = (
        df[['event_id', 'cp_id']]
        .drop_duplicates()
        .shape[0]
    )
    
    ratio = total_reco / total_cp if total_cp > 0 else 0.0
    return ratio



In [3]:
import numpy as np
import pandas as pd
import awkward as ak

from src.data_loader.Data_Loader import Data_Loader
from src.models.model_loader import model_loader
from src.clustering.clusterer import clusterer
from src.metrics.build_dataframe import build_dataframe

In [28]:
loader = Data_Loader(
    #root="/vols/cms/mm1221/geant4sim/simulations/build/Datasets/Test_Pion_2_7/Pion.5003_5004.root",
    root="/vols/cms/mm1221/geant4sim/simulations/build/Datasets/Test_EM_2_10_delta5/Electron.10003_10004.root",
    split="test",
    max_events=50,
    batch_size=1,
    shuffle=False,
    follow_batch=["x"],
)

### Loading ROOT file: /vols/cms/mm1221/geant4sim/simulations/build/Datasets/Test_EM_2_10_delta5/Electron.10003_10004.root
### Keeping only events with len(PrimaryEnergies) in (6, 7, 8, 9)
Loaded 18 events (post-filter)


/cvmfs/sft.cern.ch/lcg/views/LCG_105a_cuda/x86_64-el9-gcc11-opt/lib/python3.9/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [29]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_config = {
    "task": "contrastive",
    "hidden_dim": 64,
    "num_layers": 3,
    "dropout": 0.01,
    "k": 24,

    "contrastive_dim": 16,
    "coord_dim": 16,
    "path" : "/vols/cms/mm1221/geant4sim/scripts/training/Contrastive/runs/EM_2_10_CD16_Supcon_delta5_t01/best_model.pt"
}

model = model_loader(
    config=model_config,
    device = device
)

print(f"Model loaded successfully")


Model loaded successfully


In [30]:
import torch.nn.functional as F
all_cluster_labels = []
all_predictions = []
all_energies = []
true_labels = []

end = 10

for i, data in enumerate(loader):
    data = data.to(device)
    all_energies.append(data.x[:, 3])

    out = model(data.x, data.x_batch)
    preds = out[0]                      # (N, D) embeddings
    preds = F.normalize(preds, p=2, dim=1)
    all_predictions.append(preds)
    true_labels.append(data.assoc)


    agglomerative = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=11,
        linkage="ward",
        metric="euclidean",
    )

    preds_np = preds.detach().cpu().numpy()
    cluster_labels = agglomerative.fit_predict(preds_np)
    



    all_cluster_labels.append(cluster_labels)

    print(i)
    if i > end:
        break
        
        
total_off = 0
for i, data in enumerate(loader):
    num_reco = np.max(all_cluster_labels[i])
    num_cp = np.unique(data.assoc[i])
    offset = num_reco - num_cp
    total_off += offset
    if i > end:
        break

print(total_off)

0
1
2
3
4
5
6
7
8
9
10
11
[-12]


In [31]:
df = build_dataframe(all_cluster_labels, loader)
pur = calc_purity(df)
eff = calc_efficiency(df)
response = calc_response(df)

In [26]:
print(pur)
print(eff)
print(response)

0.9625
1.0
(1.507394, 1.4775227)


In [12]:
total_off = 0
for i, data in enumerate(loader):
    num_reco = np.max(all_cluster_labels[i])
    num_cp = np.unique(data.assoc[i])
    offset = num_reco - num_cp
    total_off += offset
    if i > 5:
        break

print(total_off)

[4]


In [32]:
df.to_csv("../dfs_table/EM_cont_16/20.csv", index=False)